# Finding_Dk_Df_from_resonance_fitting

This is the **rust-rf** edition of the upstream scikit-rf notebook. It demonstrates
preparing a one-port resonator response for typed Q-factor fitting. Rust examples use typed values and explicit `Result` handling.

The original notebook contained 2 Python code cells. They have been
replaced by a compact, executable Rust workflow instead of retaining stale Python syntax.
The copied data files and figures remain beside the notebook for extending the example.


## Kernel setup

The first cell adds the local crate and the numerical types used by the example.


In [ ]:
:dep rust-rf = { path = "../../../.." }
:dep ndarray = "0.17.2"
:dep num-complex = "0.4.6"


## Example


In [ ]:
use ndarray::{Array1, Array2, Array3};
use num_complex::Complex64;
use rust_rf::qfactor::{QFactor, ResonanceType};
use rust_rf::{Frequency, Network};

let frequency = Frequency::from_hz(Array1::linspace(0.99e9, 1.01e9, 101))
    .expect("the frequency sweep should be valid");
let scattering = Array3::from_shape_fn((101, 1, 1), |(point, _, _)| {
    let offset = (frequency.values_hz()[point] - 1.0e9) / 1.0e9;
    Complex64::new(0.1, 0.0) + Complex64::new(0.9, 0.0)
        / Complex64::new(1.0, 100.0 * offset)
});
let network = Network::new(
    frequency,
    scattering,
    Array2::from_elem((101, 1), Complex64::new(50.0, 0.0)),
)
.expect("the resonator network should be valid");
let fitter = QFactor::new(network, ResonanceType::Reflection, Some(100.0), Some(1.0e9))
    .expect("the Q-factor fitter should initialize");
let result = fitter.fit().expect("the resonance should fit");

println!("loaded Q = {:.2}; RMS error = {:.3e}", result.loaded_q, result.rms_error);


## Rust API used

- `qfactor::QFactor`
- `qfactor::ResonanceType`
- `QFactor::fit`

The complete crate reference is generated from the Rust source with
`cargo doc --all-features --open`. See `PORTING_MAP.md` at the repository root for
the detailed upstream-to-Rust coverage map.
